# 🚀 CE-EEN-B0: Two-Stage Training (Anti-Overfitting)

## 🎯 Training Strategy
This notebook uses a **two-stage training approach** to prevent overfitting:

### Stage 1: Train Head Only (Frozen Base)
- Freeze EfficientNetB0 base model
- Train only the custom classification head
- Higher learning rate (1e-3)
- 15 epochs

### Stage 2: Fine-Tune Entire Model (Unfrozen)
- Unfreeze EfficientNetB0
- Fine-tune entire model with lower LR
- Very low learning rate (1e-5)
- 25 epochs

## 🛡️ Anti-Overfitting Techniques
- **Batch Normalization**: Stabilizes training
- **Higher Dropout**: 0.5 and 0.4 (increased from 0.4 and 0.3)
- **L2 Regularization**: Weight decay on dense layers
- **Reduced Label Smoothing**: 0.05 (reduced from 0.1)
- **Data Augmentation**: Aggressive augmentation
- **Early Stopping**: Patience of 5 epochs
- **ReduceLROnPlateau**: Adaptive learning rate

---

In [ ]:
# Install visualization tool
!pip install -q livelossplot

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, mixed_precision, regularizers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from livelossplot import PlotLossesKeras

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, balanced_accuracy_score

print(f"TensorFlow Version: {tf.__version__}")
print(f"Keras Version: {keras.__version__}")

In [ ]:
# ==========================================
# 🚀 GPU CONFIGURATION
# ==========================================
gpus = tf.config.list_physical_devices('GPU')
if not gpus:
    raise RuntimeError("❌ NO GPU DETECTED! Please enable GPU in Settings.")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

mixed_precision.set_global_policy('mixed_float16')
print(f"✓ GPU Active: {len(gpus)} devices")
print("✓ Mixed Precision: ON")

# Config
SEED = 42
IMG_SIZE = 180
BATCH_SIZE = 32          # Reduced from 64 for better generalization
STAGE1_EPOCHS = 15       # Train head only
STAGE2_EPOCHS = 25       # Fine-tune entire model
STAGE1_LR = 1e-3         # Higher LR for head training
STAGE2_LR = 1e-5         # Very low LR for fine-tuning
LABEL_SMOOTH = 0.05      # Reduced from 0.1
L2_REG = 1e-4            # L2 regularization strength

BASE_DIR = Path('/kaggle/input/massive-skin-disease-balanced-dataset')
MODEL_DIR = Path('/kaggle/working/models')
MODEL_DIR.mkdir(exist_ok=True, parents=True)

tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"\n📁 Base Directory: {BASE_DIR}")
print(f"📁 Model Output Directory: {MODEL_DIR}")

## 1️⃣ Data Loading (Same as before)

In [ ]:
# Smart dataset detection
def find_image_folders(root_path, max_depth=3):
    image_extensions = {'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}
    
    def search_recursive(path, depth=0):
        if depth > max_depth:
            return None
        subdirs = [d for d in path.iterdir() if d.is_dir()]
        if not subdirs:
            return None
        folders_with_images = []
        for subdir in subdirs:
            files = list(subdir.iterdir())[:100]
            has_images = any(f.suffix in image_extensions for f in files if f.is_file())
            if has_images:
                folders_with_images.append(subdir)
        if len(folders_with_images) >= 3:
            return path
        for subdir in subdirs:
            result = search_recursive(subdir, depth + 1)
            if result:
                return result
        return None
    return search_recursive(root_path)

DATA_DIR = find_image_folders(BASE_DIR)
if DATA_DIR is None:
    raise ValueError("Could not locate dataset!")
print(f"✓ Found dataset at: {DATA_DIR}")

In [ ]:
# Load images and labels
class_folders = sorted([f for f in DATA_DIR.iterdir() if f.is_dir()])
class_names = [f.name for f in class_folders]
num_classes = len(class_names)
print(f"✓ Found {num_classes} classes")

paths, labels = [], []
image_extensions = ['.jpg', '.JPG', '.jpeg', '.JPEG', '.png', '.PNG']

for idx, folder in enumerate(class_folders):
    imgs = []
    for ext in image_extensions:
        imgs.extend(list(folder.glob(f'*{ext}')))
    if len(imgs) > 0:
        paths.extend([str(p) for p in imgs])
        labels.extend([idx] * len(imgs))

paths = np.array(paths)
labels = np.array(labels)
print(f"✓ Total Images: {len(paths):,}")
print(f"✓ Images per class (avg): {len(paths)//num_classes:,}")

In [ ]:
# Train/Val/Test Split
train_p, temp_p, train_l, temp_l = train_test_split(
    paths, labels, test_size=0.3, stratify=labels, random_state=SEED
)
val_p, test_p, val_l, test_l = train_test_split(
    temp_p, temp_l, test_size=0.5, stratify=temp_l, random_state=SEED
)

print(f"✓ Train: {len(train_p):,} | Val: {len(val_p):,} | Test: {len(test_p):,}")

## 2️⃣ Enhanced Data Pipeline with Stronger Augmentation

In [ ]:
def extract_contour_and_crop(path_bytes):
    try:
        path = path_bytes.numpy().decode('utf-8')
        img = cv2.imread(path)
        if img is None:
            return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return cv2.resize(rgb, (IMG_SIZE, IMG_SIZE))
        cnt = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(cnt)
        margin = int(0.05 * max(w, h))
        x, y = max(0, x-margin), max(0, y-margin)
        w = min(img.shape[1]-x, w+2*margin)
        h = min(img.shape[0]-y, h+2*margin)
        cropped = rgb[y:y+h, x:x+w]
        return cv2.resize(cropped, (IMG_SIZE, IMG_SIZE))
    except:
        return np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

def preprocess(path, label):
    img = tf.py_function(extract_contour_and_crop, [path], tf.uint8)
    img.set_shape([IMG_SIZE, IMG_SIZE, 3])
    img = tf.cast(img, tf.float32) / 255.0
    label = tf.one_hot(label, num_classes)
    return img, label

def augment(img, label):
    """Stronger augmentation to prevent overfitting"""
    # Random flips
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    
    # Random rotation (0, 90, 180, 270 degrees)
    img = tf.image.rot90(img, tf.random.uniform([], 0, 4, tf.int32))
    
    # Color augmentation (stronger)
    img = tf.image.random_brightness(img, 0.3)  # Increased from 0.2
    img = tf.image.random_contrast(img, 0.7, 1.3)  # Wider range
    img = tf.image.random_saturation(img, 0.7, 1.3)  # Wider range
    img = tf.image.random_hue(img, 0.1)  # NEW: Add hue variation
    
    # Random zoom (NEW)
    if tf.random.uniform([]) > 0.5:
        zoom_factor = tf.random.uniform([], 0.8, 1.0)
        img = tf.image.central_crop(img, zoom_factor)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    
    return tf.clip_by_value(img, 0, 1), label

def make_ds(paths, labels, aug=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if aug:
        ds = ds.shuffle(20000, seed=SEED)
    ds = ds.map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if aug:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(train_p, train_l, aug=True)
val_ds = make_ds(val_p, val_l)
test_ds = make_ds(test_p, test_l)
print("✓ Data Pipeline with Enhanced Augmentation")

## 3️⃣ Improved Model with Batch Normalization & Regularization

In [ ]:
def build_model(n_classes):
    """Build model with BatchNorm and L2 regularization"""
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    
    # EfficientNetB0 base
    base = EfficientNetB0(include_top=False, weights='imagenet', input_tensor=inp)
    base.trainable = False  # 🔒 FROZEN initially
    
    x = base.output
    
    # Enhanced Custom Head with BatchNorm
    x = layers.Conv2D(64, 3, padding='same', kernel_regularizer=regularizers.l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)  # NEW: Batch Normalization
    x = layers.Activation('relu')(x)
    
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)  # Increased from 0.4
    
    x = layers.Dense(256, kernel_regularizer=regularizers.l2(L2_REG))(x)
    x = layers.BatchNormalization()(x)  # NEW: Batch Normalization
    x = layers.Activation('relu')(x)
    x = layers.Dropout(0.4)(x)  # Increased from 0.3
    
    # Output layer (Float32)
    out = layers.Dense(n_classes, activation='softmax', dtype='float32')(x)
    
    return models.Model(inp, out, name='CE_EEN_B0_TwoStage')

model = build_model(num_classes)
print(f"\n✓ Model built with {num_classes} classes")
print(f"✓ Base model frozen: {not model.layers[1].trainable}")
print(f"✓ Trainable params: {model.count_params():,}")

## 4️⃣ STAGE 1: Train Head Only (Frozen Base)

In [ ]:
print("\n" + "="*70)
print("STAGE 1: Training Classification Head (Base Frozen)")
print("="*70)

# Compile for Stage 1
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=STAGE1_LR),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
    metrics=['accuracy']
)

# Callbacks for Stage 1
stage1_callbacks = [
    ModelCheckpoint(
        str(MODEL_DIR / 'stage1_best.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    PlotLossesKeras(),
    keras.callbacks.CSVLogger(str(MODEL_DIR / 'stage1_log.csv'))
]

print(f"\nStage 1 Config:")
print(f"  Epochs: {STAGE1_EPOCHS}")
print(f"  Learning Rate: {STAGE1_LR}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Base Model: FROZEN")
print(f"\n🚀 Starting Stage 1 training...\n")

history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE1_EPOCHS,
    callbacks=stage1_callbacks,
    verbose=1
)

print("\n✓ Stage 1 Complete!")

## 5️⃣ STAGE 2: Fine-Tune Entire Model (Unfrozen)

In [ ]:
print("\n" + "="*70)
print("STAGE 2: Fine-Tuning Entire Model (Base Unfrozen)")
print("="*70)

# Load best model from Stage 1
model = keras.models.load_model(str(MODEL_DIR / 'stage1_best.keras'))

# Unfreeze base model
model.layers[1].trainable = True
print(f"✓ Base model unfrozen")
print(f"✓ Total trainable params: {model.count_params():,}")

# Recompile with VERY LOW learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=STAGE2_LR),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTH),
    metrics=['accuracy']
)

# Callbacks for Stage 2
stage2_callbacks = [
    ModelCheckpoint(
        str(MODEL_DIR / 'stage2_best.keras'),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    PlotLossesKeras(),
    keras.callbacks.CSVLogger(str(MODEL_DIR / 'stage2_log.csv'))
]

print(f"\nStage 2 Config:")
print(f"  Epochs: {STAGE2_EPOCHS}")
print(f"  Learning Rate: {STAGE2_LR} (very low!)")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Base Model: UNFROZEN")
print(f"\n🚀 Starting Stage 2 training...\n")

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=STAGE2_EPOCHS,
    callbacks=stage2_callbacks,
    verbose=1
)

print("\n✓ Stage 2 Complete!")

## 6️⃣ Evaluation & Saving

In [ ]:
# Load best model from Stage 2
print("Loading best model from Stage 2...")
model = keras.models.load_model(str(MODEL_DIR / 'stage2_best.keras'))

# Evaluate
print("\nEvaluating on test set...")
loss, acc = model.evaluate(test_ds, verbose=1)
print(f"\n{'='*60}")
print(f"🏆 Test Loss: {loss:.4f}")
print(f"🏆 Test Accuracy: {acc:.4f} ({acc*100:.2f}%)")
print(f"{'='*60}")

In [ ]:
# Classification Report
print("\nGenerating classification report...")
y_true, y_pred = [], []

for imgs, labs in test_ds:
    preds = model.predict(imgs, verbose=0)
    y_true.extend(np.argmax(labs.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print("\n" + "="*80)
print("CLASSIFICATION REPORT")
print("="*80)
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

bal_acc = balanced_accuracy_score(y_true, y_pred)
print(f"\nBalanced Accuracy: {bal_acc:.4f} ({bal_acc*100:.2f}%)")

In [ ]:
# Save final model
print("\nSaving final model...")
model.save(str(MODEL_DIR / 'final_model.keras'))
np.save(str(MODEL_DIR / 'classes.npy'), class_names)

print("\n✓ All models saved:")
print(f"  - Stage 1 best: {MODEL_DIR / 'stage1_best.keras'}")
print(f"  - Stage 2 best: {MODEL_DIR / 'stage2_best.keras'}")
print(f"  - Final model: {MODEL_DIR / 'final_model.keras'}")
print(f"  - Classes: {MODEL_DIR / 'classes.npy'}")

## 7️⃣ Training History Visualization

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Load logs
stage1_log = pd.read_csv(str(MODEL_DIR / 'stage1_log.csv'))
stage2_log = pd.read_csv(str(MODEL_DIR / 'stage2_log.csv'))

# Accuracy plot
axes[0].plot(stage1_log['accuracy'], label='Stage 1 Train', linewidth=2)
axes[0].plot(stage1_log['val_accuracy'], label='Stage 1 Val', linewidth=2)
axes[0].plot(range(len(stage1_log), len(stage1_log) + len(stage2_log)), 
             stage2_log['accuracy'], label='Stage 2 Train', linewidth=2)
axes[0].plot(range(len(stage1_log), len(stage1_log) + len(stage2_log)), 
             stage2_log['val_accuracy'], label='Stage 2 Val', linewidth=2)
axes[0].axvline(x=len(stage1_log), color='red', linestyle='--', label='Unfreeze Point')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss plot
axes[1].plot(stage1_log['loss'], label='Stage 1 Train', linewidth=2)
axes[1].plot(stage1_log['val_loss'], label='Stage 1 Val', linewidth=2)
axes[1].plot(range(len(stage1_log), len(stage1_log) + len(stage2_log)), 
             stage2_log['loss'], label='Stage 2 Train', linewidth=2)
axes[1].plot(range(len(stage1_log), len(stage1_log) + len(stage2_log)), 
             stage2_log['val_loss'], label='Stage 2 Val', linewidth=2)
axes[1].axvline(x=len(stage1_log), color='red', linestyle='--', label='Unfreeze Point')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(MODEL_DIR / 'training_history.png'), dpi=150, bbox_inches='tight')
plt.show()

print("✓ Training history saved")

## 8️⃣ Inference Demo

In [ ]:
# Load model for inference
loaded_model = keras.models.load_model(str(MODEL_DIR / 'final_model.keras'))
loaded_classes = np.load(str(MODEL_DIR / 'classes.npy'))

def predict_image_path(image_path):
    if not os.path.exists(image_path):
        print(f"❌ File not found: {image_path}")
        return
    
    img = cv2.imread(image_path)
    if img is None:
        print("❌ Could not read image")
        return
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, thresh = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contours:
        cnt = max(contours, key=cv2.contourArea)
        x, y, w, h = cv2.boundingRect(cnt)
        margin = int(0.05 * max(w, h))
        x, y = max(0, x-margin), max(0, y-margin)
        w = min(img.shape[1]-x, w+2*margin)
        h = min(img.shape[0]-y, h+2*margin)
        img_crop = img_rgb[y:y+h, x:x+w]
    else:
        img_crop = img_rgb
    
    img_resized = cv2.resize(img_crop, (IMG_SIZE, IMG_SIZE))
    img_batch = np.expand_dims(img_resized / 255.0, axis=0)
    
    preds = loaded_model.predict(img_batch, verbose=0)
    idx = np.argmax(preds)
    class_name = loaded_classes[idx]
    conf = preds[0][idx]
    
    plt.figure(figsize=(8, 8))
    plt.imshow(img_crop)
    plt.title(f"Prediction: {class_name}\nConfidence: {conf:.2%}",
              fontsize=16, color='green' if conf > 0.7 else 'orange', fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    top3_idx = np.argsort(preds[0])[-3:][::-1]
    print("\nTop 3 Predictions:")
    for i, idx in enumerate(top3_idx, 1):
        print(f"  {i}. {loaded_classes[idx]}: {preds[0][idx]:.2%}")

print("✓ Inference function ready")
print("Usage: predict_image_path('path/to/image.jpg')")